In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import joblib

print("--- LiveCare Anomaly Detection Pipeline Initialization ---\n")

# ==========================================
# 1. LOAD DATASET
# ==========================================
print("1. Loading production dataset...")
df = pd.read_csv('livecare_production_dataset.csv')

# ==========================================
# 2. SPLIT HEALTHY & SICK DATA
# ==========================================
print("2. Segregating healthy baseline from anomaly events...")

df_healthy = df[df['status'] == 1.0]
df_sick = df[df['status'] == -1.0]

# 🔥 IMPORTANT: Shuffle to avoid bias
df_healthy = df_healthy.sample(frac=1, random_state=42)

train_size = int(len(df_healthy) * 0.8)

# ==========================================
# FEATURE ORDER (CRITICAL)
# ==========================================
feature_order = [
    'heart_rate','temperature_body','spo2',
    'accel_x','accel_y','accel_z',
    'gyro_pitch','temperature_ambient','env_humidity'
]

# ==========================================
# TRAIN SET (ONLY HEALTHY)
# ==========================================
X_train = df_healthy.iloc[:train_size].drop(columns=['timestamp', 'status'])
X_train = X_train[feature_order]

# ==========================================
# TEST SET (HEALTHY + SICK)
# ==========================================
test_healthy = df_healthy.iloc[train_size:]
df_test = pd.concat([test_healthy, df_sick]).sample(frac=1, random_state=42)

X_test = df_test.drop(columns=['timestamp', 'status'])
X_test = X_test[feature_order]

y_test_true = df_test['status']

# ==========================================
# 3. FEATURE SCALING (MANDATORY)
# ==========================================
print("3. Applying Standard Scaling...")

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 4. TRAIN ISOLATION FOREST
# ==========================================
print("4. Training Isolation Forest...")

iso_forest = IsolationForest(
    n_estimators=200,
    contamination='auto',   # 🔥 improved
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_train_scaled)

# ==========================================
# 5. PREDICT & EVALUATE
# ==========================================
print("5. Evaluating model...\n")

y_pred = iso_forest.predict(X_test_scaled)

print("======================================================")
print("              MODEL CLASSIFICATION REPORT             ")
print("======================================================")
print(classification_report(
    y_test_true,
    y_pred,
    target_names=['Anomaly (-1) / Sick', 'Healthy (1)']
))

print("======================================================")
print("                   CONFUSION MATRIX                   ")
print("======================================================")

cm = confusion_matrix(y_test_true, y_pred, labels=[-1, 1])

print(f"True Anomalies Caught:      {cm[0][0]}")
print(f"Missed Anomalies:           {cm[0][1]}")
print(f"False Alarms:               {cm[1][0]}")
print(f"Correct Healthy Readings:   {cm[1][1]}")

print("==========================================")

# 6. EXPORT MODEL + SCALER
print("6. Saving model and scaler...")

joblib.dump(iso_forest, 'livecare_isolation_forest.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("✅ Model and scaler saved successfully!")

--- LiveCare Anomaly Detection Pipeline Initialization ---

1. Loading production dataset...
2. Segregating healthy baseline from anomaly events...
3. Applying Standard Scaling...
4. Training Isolation Forest...
5. Evaluating model...

              MODEL CLASSIFICATION REPORT             
                     precision    recall  f1-score   support

Anomaly (-1) / Sick       0.76      1.00      0.86      1700
        Healthy (1)       1.00      0.80      0.89      2660

           accuracy                           0.88      4360
          macro avg       0.88      0.90      0.88      4360
       weighted avg       0.91      0.88      0.88      4360

                   CONFUSION MATRIX                   
True Anomalies Caught:      1700
Missed Anomalies:           0
False Alarms:               536
Correct Healthy Readings:   2124
6. Saving model and scaler...
✅ Model and scaler saved successfully!
